In [ ]:
import boto3
import csv
import io
from datetime import datetime
from botocore.exceptions import ClientError

# --- CONFIGURACIÓN ---
REGION = 'us-west-2'
BUCKET_RAW = 'bucket_name-raw'  # Reemplaza con tu bucket real
BUCKET_PROCESSED = 'bucket_name' # Reemplaza con tu bucket real
ARCHIVO_LOCAL = './mis_archivos/datos_ventas.csv'

# Inicializar el cliente de S3
s3_client = boto3.client('s3', region_name=REGION)

In [10]:
# 1. Crear Infraestructura (buckets)
def preparar_buckets():
    buckets = [BUCKET_RAW, BUCKET_PROCESSED]
    todas_creadas = True  # Inicializamos la variable de control
    
    for b in buckets:
        try:
            if REGION == 'us-east-1':
                s3_client.create_bucket(Bucket=b)
            else:
                s3_client.create_bucket(
                    Bucket=b, 
                    CreateBucketConfiguration={'LocationConstraint': REGION}
                )
            print(f"Bucket listo: {b}")
            
        except ClientError as e:
            error_code = e.response['Error']['Code']
            if error_code == 'BucketAlreadyOwnedByYou':
                print(f"El bucket '{b}' ya te pertenece.")
            elif error_code == 'BucketAlreadyExists':
                print(f"Error: El nombre '{b}' ya está en uso por otro usuario a nivel global.")
                todas_creadas = False # Ahora sí está definida
            else:
                print(f"Error inesperado al crear '{b}': {e}")
                todas_creadas = False
                
    return todas_creadas # Retorna el estado real de la operación

In [5]:
# 2. Lógica de Procesamiento CSV
def procesar_metricas_ventas(contenido_csv):
    """Analiza datos del CSV en memoria."""
    f = io.StringIO(contenido_csv)
    lector = csv.DictReader(f)
    
    total_ingresos = 0
    total_articulos = 0
    categorias = []
    
    for fila in lector:
        cantidad = int(fila['cantidad'])
        precio = float(fila['precio_unitario'])
        
        total_ingresos += (cantidad * precio)
        total_articulos += cantidad
        categorias.append(fila['categoria'])
    
    # Obtener categoría más repetida
    categoria_top = max(set(categorias), key=categorias.count)
    
    # Generar reporte formateado
    reporte = (
        f"--- RESUMEN DE VENTAS DIARIAS ---\n"
        f"Fecha de Procesamiento: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n"
        f"----------------------------------\n"
        f"Ingresos Totales: ${total_ingresos:,.2f}\n"
        f"Total de Artículos Vendidos: {total_articulos}\n"
        f"Categoría más Vendida: {categoria_top}\n"
        f"Promedio por Transacción: ${total_ingresos/10:,.2f}\n"
    )
    return reporte

In [6]:
# 3. Pipeline Principal
def ejecutar_pipeline():
    print("Iniciando Pipeline ETL...")
    
    # A. Ingesta (Upload)
    s3_client.upload_file(ARCHIVO_LOCAL, BUCKET_RAW, f"raw/{ARCHIVO_LOCAL}")
    
    # B. Transformación (Read & Process)
    obj = s3_client.get_object(Bucket=BUCKET_RAW, Key=f"raw/{ARCHIVO_LOCAL}")
    cuerpo = obj['Body'].read().decode('utf-8')
    reporte_final = procesar_metricas_ventas(cuerpo)
    
    # C. Carga (Output)
    nombre_reporte = f"reportes/resultado_{datetime.now().strftime('%H%M')}.txt"
    s3_client.put_object(Bucket=BUCKET_PROCESSED, Key=nombre_reporte, Body=reporte_final)
    
    print(f"Proceso terminado. Reporte guardado como: {nombre_reporte}")
    print("\n--- Vista Previa del Reporte ---")
    print(reporte_final)


In [13]:
if __name__ == "__main__":
    preparar_buckets()
    ejecutar_pipeline()

Bucket listo: s3-vd-ventas-raw-vv
Bucket listo: s3-vd-ventas-reportes-vv
Iniciando Pipeline ETL...
Proceso terminado. Reporte guardado como: reportes/resultado_2353.txt

--- Vista Previa del Reporte ---
--- RESUMEN DE VENTAS DIARIAS ---
Fecha de Procesamiento: 2026-04-13 23:53
----------------------------------
Ingresos Totales: $2,252.48
Total de Artículos Vendidos: 19
Categoría más Vendida: Electronica
Promedio por Transacción: $225.25

